# Notebook 02 - Prompt Engineering

## Objective

This notebook explores different prompt engineering strategies to generate product
descriptions from the cleaned metadata produced in Notebook 01.

Three distinct prompt styles are tested and evaluated:
- **Short** : concise, factual 2-3 sentence descriptions
- **Marketing** : persuasive, engaging 3-4 sentence descriptions
- **Technical** : precise, specification-focused 3-4 sentence descriptions

The goal is to validate prompt quality on a sample before running the full generation
pipeline on all 800 products. The best-performing style will be identified in
Notebook 03 via BLEU, ROUGE-L and cosine similarity metrics.

In [ ]:
!pip install transformers accelerate sentencepiece bitsandbytes -q

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

import json
import os

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
Mounted at /content/drive


## 1 - Load Data & Cache

In [ ]:
df = pd.read_csv("clean_products_800.csv")
print(f"Dataset shape : {df.shape}")
print(f"Columns       : {df.columns.tolist()}")
df.head()

,title,description,brand,category,price
0,"Nikon DSLR Bag with Online Class Camera Case, ...",Protect your valuable photography equipment wi...,Nikon,Camera Cases,$44.95
1,BosStrap One Piece (OP) Sling Strap (Black),A BosTail replaces the tripod fitting supplied...,BosStrap,Bag & Case Accessories,$39.95
2,I3ePro BP-CMIC1 X-Series Mini Shotgun Condense...,The BP-CMIC1 Stereo Microphone is ideal for vi...,PIXI-GEAR,Microphones,$34.99
3,Targus Truss Case/Stand for 10.1-Inch Acer Ico...,The Targus Truss Case/Stand for Acer Iconia is...,Targus,Cases,NaN
4,JETBeam i4 PRO Intelligent Charger enhanced V3...,The popular i4 charger has been enhanced for b...,JETBeam,Chargers & Adapters,NaN


In [ ]:
cache_path = "/content/drive/MyDrive/partial_results.jsonl"

if os.path.exists(cache_path):
    cache = {
        json.loads(l)["title"]: json.loads(l)
        for l in open(cache_path, "r", encoding="utf-8")
    }
    print(f"Cache loaded : {len(cache)} entries")
else:
    cache = {}
    print("No cache found, starting fresh.")

## 2 - LLM Setup

We use **Mistral-7B-Instruct-v0.3**, a 7-billion parameter instruction-tuned model.

**4-bit quantization** is applied via `BitsAndBytesConfig` to reduce VRAM usage
from ~14GB (bfloat16) to ~5GB, making it compatible with free-tier Colab GPUs
with no significant quality loss for this task.

The `llm()` function uses `apply_chat_template()` to correctly format prompts
with Mistral's `[INST]...[/INST]` instruction tokens. Without this, the model
does not recognize the input as an instruction and produces degraded outputs.

`do_sample=True` with `temperature=0.7` is used for generation (creative
variation), while `do_sample=False` is reserved for deterministic tasks.

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
def llm(prompt, max_tokens=200, do_sample=True):
    """
    Generate text from a prompt using Mistral-7B-Instruct.

    Args:
        prompt      : user instruction string
        max_tokens  : maximum number of new tokens to generate
        do_sample   : if True, uses temperature sampling; if False, greedy decoding

    Returns:
        Generated text as a stripped string.
    """
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )

    # Handle BatchEncoding vs raw tensor depending on transformers version
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids.input_ids

    input_ids = input_ids.to(model.device)

    # Attention mask set manually to avoid warning when pad_token == eos_token
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_tokens,
        do_sample=do_sample,
        temperature=0.7 if do_sample else None,
        top_p=0.9 if do_sample else None,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 3 - Prompt Design

Three prompts are defined, each targeting a different writing style.

**Design principles applied:**
- Input is limited to `title`, `brand`, `category`, `price` — the only available metadata
- All prompts explicitly forbid inventing features or specifications not present in the data
- Negative examples are provided inline to prevent recurring hallucination patterns
  (e.g. "officially licensed", "steel construction") — Mistral 7B responds better to
  concrete examples than abstract rules
- Marketing prompt allows stylistic freedom while prohibiting verifiable factual claims
- All prompts forbid bullet points and emojis for clean, Amazon-style prose output

**`max_tokens` rationale:**
- Short : 160 tokens (~2-3 sentences with margin)
- Marketing / Technical : 200 tokens (~3-4 sentences with margin for longer titles)

In [ ]:
prompt_short = """
Write a short factual product description using ONLY the information below.
Do not invent anything. Use only what is provided
Do not mention missing data.
For example, never write "ensuring quality", "reliable", "compatible with most" unless stated in the data.

Title: {title}
Brand: {brand}
Category: {category}
Price: {price}

Write 2–3 factual sentences. No bullet points. No emojis.
"""

prompt_marketing = """
Write a persuasive marketing product description using ONLY the information below.
You may use engaging style and tone.
Do NOT invent product features or technical specifications.
Do NOT make verifiable claims that are not supported by the data.

Title: {title}
Brand: {brand}
Category: {category}
Price: {price}

Write 3–4 engaging sentences. No bullet points. No emojis.
"""

prompt_technical = """
Write a clear technical product description using ONLY the information below.
Do not invent specifications, materials, or features.
Do not make verifiable claims that are not supported by the data.
Do not mention missing data or absent specifications.
For example, never write "steel construction", "integrated microphone", "embroidered logos" unless stated in the data.

Title: {title}
Brand: {brand}
Category: {category}
Price: {price}

Write 3–4 technical sentences. No bullet points. No emojis.
"""

In [ ]:
def generate_all_versions(row):
    """
    Generate short, marketing and technical descriptions for a single product row.

    Args:
        row : pandas Series with fields title, brand, category, price

    Returns:
        dict with all input fields and 3 generated descriptions
    """
    title    = row["title"]
    brand    = row["brand"]
    category = row["category"]
    price    = row["price"]

    short_desc = llm(prompt_short.format(
        title=title, brand=brand, category=category, price=price
    ), max_tokens=160, do_sample=True)

    marketing_desc = llm(prompt_marketing.format(
        title=title, brand=brand, category=category, price=price
    ), max_tokens=200, do_sample=True)

    technical_desc = llm(prompt_technical.format(
        title=title, brand=brand, category=category, price=price
    ), max_tokens=200, do_sample=True)

    return {
        "title":                 title,
        "brand":                 brand,
        "category":              category,
        "price":                 price,
        "short_description":     short_desc,
        "marketing_description": marketing_desc,
        "technical_description": technical_desc,
    }

## 4 - Prompt Validation

Before running the full pipeline on 800 products, we validate prompt quality
on 5 products sampled at regular intervals across the dataset.

This step ensures that:
- No systematic hallucinations are present
- Output format is respected (sentence count, no bullet points)
- No truncations occur within `max_tokens` budget

In [ ]:
sample_indices = [0, 150, 300, 500, 700]

for i in sample_indices:
    row = df.iloc[i]
    out = generate_all_versions(row)
    print(f"\n{'='*60}")
    print(f"PRODUCT  : {row['title']}")
    print(f"Brand: {row['brand']} | Category: {row['category']} | Price: {row['price']}")
    print(f"\n--- SHORT ---\n{out['short_description']}")
    print(f"\n--- MARKETING ---\n{out['marketing_description']}")
    print(f"\n--- TECHNICAL ---\n{out['technical_description']}")

### Validation Results — Round 4 (Final)

#### Hallucinations

| Product | Short | Marketing | Technical |
|---|---|---|---|
| NCAA Bearcats | ✅ | ✅ | ⚠️ "access to all ports and buttons" (inferred but common) |
| Samsung HDD | ✅ | ⚠️ "sleek design" (style, acceptable) | ✅ |
| Eforcity Speaker | ✅ | ✅ | ✅ |
| SteelSeries | ✅ | ⚠️ "all-night comfort" (unverifiable) | ❌ "steel construction", "separate microphone outputs" (invented) |
| Addonics | ✅ | ⚠️ "speed and reliability of SATA" (unverifiable) | ✅ |

#### Format & Truncations

| Product | Short | Marketing | Technical |
|---|---|---|---|
| NCAA Bearcats | ✅ | ✅ | ✅ |
| Samsung HDD | ✅ | ✅ | ✅ |
| Eforcity Speaker | ✅ | ✅ | ✅ |
| SteelSeries | ✅ | ✅ | ✅ |
| Addonics | ✅ | ✅ | ✅ |

#### Verdict

| Product | Status | Note |
|---|---|---|
| NCAA Bearcats | ✅ | |
| Samsung HDD | ✅ | |
| Eforcity Speaker | ✅ | |
| SteelSeries | ⚠️ | Isolated technical hallucination — model associates "Steel" brand with steel material |
| Addonics | ✅ | |

### Observation

Residual hallucinations in the technical style are linked to word associations
in the model's training data (e.g. "SteelSeries" → "steel construction").
This is an intrinsic limitation of Mistral 7B on sparse input and cannot be
eliminated through prompt engineering alone. The overall hallucination rate
is estimated at ~10-15%, which is acceptable for a fine-tuning training dataset.

## 5 - Full Generation Pipeline

In [ ]:
results = []
errors  = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    if row["title"] in cache:
        out = cache[row["title"]]

    else:
        try:
            out = generate_all_versions(row)
            with open(cache_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
            cache[row["title"]] = out
        except Exception as e:
            errors.append({"title": row["title"], "error": str(e)})
            continue

    results.append(out)

print(f"✅ {len(results)} generated | ❌ {len(errors)} errors")

  0%|          | 0/800 [00:00<?, ?it/s]

✅ 800 générés | ❌ 0 erreurs


In [ ]:
with open("/content/drive/MyDrive/dataset.jsonl", "w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ Export complete : {len(results)} products")

✅ Export terminé : 800 produits


In [ ]:
with open("/content/drive/MyDrive/dataset.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total lines    : {len(lines)}")

corrupt = []
for i, line in enumerate(lines):
    try:
        json.loads(line)
    except json.JSONDecodeError:
        corrupt.append(i)

print(f"Corrupt lines  : {len(corrupt)}")
print(f"Columns        : {list(json.loads(lines[0]).keys())}")

Nombre de lignes : 800
Lignes corrompues : 0
dict_keys(['title', 'brand', 'category', 'price', 'short_description', 'marketing_description', 'technical_description'])


## 6 - Summary

| Step | Decision | Rationale |
|---|---|---|
| Model | Mistral-7B-Instruct-v0.3 | Best open-source instruct model at 7B scale |
| Quantization | 4-bit (bitsandbytes) | Reduces VRAM from ~14GB to ~5GB, no quality loss |
| Chat template | `apply_chat_template()` | Required for correct [INST] formatting |
| Prompt styles | Short / Marketing / Technical | Cover the 3 main Amazon description formats |
| Hallucination control | Negative examples in prompt | More effective than abstract rules for 7B models |
| max_tokens | 160 (short) / 200 (marketing, technical) | Prevents truncation with safety margin |
| Caching | JSONL append mode | Crash-safe, resumable generation |

**800 products generated, 0 corrupt lines.**

The dataset is ready for evaluation in **Notebook 03**, where BLEU, ROUGE-L
and cosine similarity scores will determine which style best approximates
original Amazon descriptions and will serve as the fine-tuning target.